In [1]:
# %% [code]
"""
Feature Set Comparison: Catch22 | ROCKET | MiniRocket | MultiRocket | Hydra | TSFresh
on 15 MTSC datasets.

Design: raw aeon TRANSFORMERS → 2D feature matrix → one shared RF classifier.
This gives a fair comparison across all feature sets because every method
uses exactly the same classifier (RandomForestClassifier n_estimators=100).

Per-fold protocol (zero leakage):
    trf = FACTORY[name]()           # fresh transformer each fold
    trf.fit(X_train)                # learns params on train fold only
    X_train_t = trf.transform(X_train)
    X_val_t   = trf.transform(X_val)   # applies fitted params, no fit on val
    rf.fit(X_train_t, y_train)
    rf.predict(X_val_t)

Transformer API confirmed from aeon docs (v1.x):
    Rocket(n_kernels, random_state)          fit(X) → transform(X) → 2D array
    MiniRocket(n_kernels, random_state)      fit(X) → transform(X) → 2D array
    MultiRocket(n_kernels, random_state)     fit(X) → transform(X) → 2D array
    HydraTransformer(n_kernels, n_groups)    fit(X) → transform(X) → 2D array
    Catch22(catch24, replace_nans)           fit(X) → transform(X) → 2D array
    TSFresh(default_fc_parameters)           fit(X) → transform(X) → 2D array

Install:
    pip install aeon tsfresh
"""

# ── Auto-install ──────────────────────────────────────────────────────────────
try:
    import aeon
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "aeon", "tsfresh"])

import warnings
import time
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score

from aeon.datasets import load_classification
from aeon.transformations.collection.convolution_based import (
    Rocket,
    MiniRocket,
    MultiRocket,
    HydraTransformer,
)
from aeon.transformations.collection.feature_based import Catch22, TSFresh

warnings.filterwarnings("ignore")


# ── Datasets ──────────────────────────────────────────────────────────────────
DATASETS = [
    "ArticularyWordRecognition",
    "AtrialFibrillation",
    "BasicMotions",
    "ERing",
    "HandMovementDirection",
    "Handwriting",
    "Libras",
    "LSST",
    "NATOPS",
    "PenDigits",
    "RacketSports",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "StandWalkJump",
    "UWaveGestureLibrary",
]

FEATURE_SETS = ["catch22", "rocket", "minirocket", "multirocket", "hydra", "tsfresh"]

# ── Detect aeon version — kernel param name changed in v1.0 ──────────────────
# v0.x: num_kernels  |  v1.0+: n_kernels
from packaging.version import Version
import aeon as _aeon
_AEON_V1 = Version(_aeon.__version__) >= Version("1.0.0")
_KERNEL_PARAM = "n_kernels" if _AEON_V1 else "num_kernels"
print(f"aeon version: {_aeon.__version__}  →  using '{_KERNEL_PARAM}' param")

# ── Transformer factories — one fresh instance per CV fold ────────────────────
FACTORIES = {
    "catch22":     lambda: Catch22(catch24=True, replace_nans=True),
    "rocket":      lambda: Rocket(**{_KERNEL_PARAM: 1_000}, random_state=42),
    "minirocket":  lambda: MiniRocket(**{_KERNEL_PARAM: 1_000}, random_state=42),
    "multirocket": lambda: MultiRocket(**{_KERNEL_PARAM: 1_000}, random_state=42),
    "hydra":       lambda: HydraTransformer(n_kernels=8, n_groups=64, random_state=42),
    "tsfresh":     lambda: TSFresh(
                       default_fc_parameters="efficient",
                       n_jobs=-1,
                       show_warnings=False,
                   ),
}


# ── Cleaning helper ───────────────────────────────────────────────────────────
def clean(X_feat):
    """Cast to float64; median-impute NaN / Inf (tsfresh can produce these)."""
    if not isinstance(X_feat, np.ndarray):
        X_feat = np.array(X_feat, dtype=float)
    X_feat = X_feat.astype(float)
    if not np.all(np.isfinite(X_feat)):
        X_feat = np.where(np.isfinite(X_feat), X_feat, np.nan)
        X_feat = SimpleImputer(strategy="median").fit_transform(X_feat)
    return X_feat


# ── CV — transformer fit inside loop, shared RF ───────────────────────────────
def evaluate_with_cv(name, X, y):
    """
    KFold(n_splits=5, shuffle=True, random_state=42).
    Each fold:
      - fresh transformer fitted on X_train only
      - 2D feature matrices produced for train and val
      - RandomForestClassifier(n_estimators=100, random_state=42) on train
    n_features taken from fold 0 directly from the array shape.
    """
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    acc_scores, f1_scores = [], []
    n_features = -1

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh transformer — fits on train fold only, no leakage
        trf = FACTORIES[name]()
        trf.fit(X_train)
        X_train_t = clean(trf.transform(X_train))
        X_val_t   = clean(trf.transform(X_val))

        # n_features straight from the array shape — no attribute introspection
        if fold_idx == 0:
            n_features = X_train_t.shape[1]

        # Same RF for every feature set — fair comparison
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_train_t, y_train)
        pred = rf.predict(X_val_t)

        acc_scores.append(accuracy_score(y_val, pred))
        f1_scores.append(f1_score(y_val, pred, average="weighted"))

    return (
        float(np.mean(acc_scores)),
        float(np.std(acc_scores)),
        float(np.mean(f1_scores)),
        float(np.std(f1_scores)),
        n_features,
    )


# ── Per-dataset runner ────────────────────────────────────────────────────────
def run_dataset(dataset_name):
    X, y = load_classification(dataset_name)
    n_samples, n_channels, series_len = X.shape
    print(f"  samples={n_samples}  channels={n_channels}  "
          f"length={series_len}  classes={len(np.unique(y))}")

    metrics_row  = {"dataset": dataset_name}
    features_row = {"dataset": dataset_name}

    for name in FEATURE_SETS:
        try:
            print(f"    {name:<12} ...", end="  ", flush=True)
            t0 = time.perf_counter()
            acc_m, acc_s, f1_m, f1_s, n_feat = evaluate_with_cv(name, X, y)
            elapsed = time.perf_counter() - t0

            print(f"acc={acc_m:.4f}±{acc_s:.4f}  "
                  f"f1={f1_m:.4f}±{f1_s:.4f}  "
                  f"feat={n_feat}  t={elapsed:.1f}s")

            metrics_row[f"{name}_accuracy"]    = f"{acc_m:.6f}±{acc_s:.6f}"
            metrics_row[f"{name}_f1"]          = f"{f1_m:.6f}±{f1_s:.6f}"
            features_row[f"{name}_n_features"] = n_feat

        except Exception as exc:
            print(f"ERROR: {exc}")
            metrics_row[f"{name}_accuracy"]    = "NaN"
            metrics_row[f"{name}_f1"]          = "NaN"
            features_row[f"{name}_n_features"] = 0

    return metrics_row, features_row


# ── Main ──────────────────────────────────────────────────────────────────────
def run_all():
    all_metrics  = {}
    all_features = {}

    for dataset in DATASETS:
        print(f"\n{'='*70}")
        print(f"Processing dataset: {dataset}")
        print(f"{'='*70}")
        try:
            metrics_row, features_row = run_dataset(dataset)
        except Exception as exc:
            print(f"  [SKIP] {dataset}: {exc}")
            metrics_row  = {"dataset": dataset}
            features_row = {"dataset": dataset}
            for name in FEATURE_SETS:
                metrics_row[f"{name}_accuracy"]    = "NaN"
                metrics_row[f"{name}_f1"]          = "NaN"
                features_row[f"{name}_n_features"] = 0

        all_metrics[dataset]  = metrics_row
        all_features[dataset] = features_row

    metrics_df  = pd.DataFrame.from_dict(all_metrics,  orient="index")
    features_df = pd.DataFrame.from_dict(all_features, orient="index")

    metrics_df = metrics_df.reindex(
        columns=["dataset"] + [c for n in FEATURE_SETS
                                for c in (f"{n}_accuracy", f"{n}_f1")],
        fill_value="NaN",
    )
    features_df = features_df.reindex(
        columns=["dataset"] + [f"{n}_n_features" for n in FEATURE_SETS],
        fill_value=0,
    )

    timestamp     = datetime.now().strftime("%Y%m%d_%H%M%S")
    metrics_file  = f"classification_metrics_results_{timestamp}.csv"
    features_file = f"classification_feature_counts_{timestamp}.csv"

    metrics_df.to_csv(metrics_file,  index=False)
    features_df.to_csv(features_file, index=False)

    print(f"\n{'='*70}")
    print(f"Results saved to:")
    print(f"  Metrics       : {metrics_file}")
    print(f"  Feature counts: {features_file}")
    print(f"{'='*70}")
    print(f"\nTotal datasets attempted : {len(DATASETS)}")
    print(f"Successfully processed   : {metrics_df['dataset'].notna().sum()}")

    print("\nMean accuracy per feature set:")
    for name in FEATURE_SETS:
        vals = []
        for v in metrics_df[f"{name}_accuracy"]:
            try:
                vals.append(float(str(v).split("±")[0]))
            except Exception:
                pass
        if vals:
            print(f"  {name:<12}: {np.mean(vals):.4f}")

    print("\nFirst few rows of metrics:")
    print(metrics_df.head().to_string())
    print("\nFirst few rows of feature counts:")
    print(features_df.head().to_string())

    return metrics_df, features_df


if __name__ == "__main__":
    metrics_df, features_df = run_all()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 51.8 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
aeon version: 1.3.0  →  using 'n_kernels' param

Processing dataset: AtrialFibrillation
  samples=30  channels=2  length=640  classes=3
    catch22      ...  acc=0.2333±0.1700  f1=0.2311±0.1850  feat=48  t=22.7s
    rocket       ...  acc=0.2333±0.0816  f1=0.2046±0.1197  feat=2000  t=8.8s
    minirocket   ...  acc=0.1333±0.1247  f1=0.1156±0.1065  feat=924  t=21.1s
    multirocket  ...  acc=0.1667±0.1054  f1=0.1484±0.1092  feat=7392  t=39.4s
    hydra        ...  acc=0.1000±0.0816  f1=0.0689±0.0815  feat=7168  t=5.0s
    tsfresh      ...  acc=0.1667±0.1054  f1=0.1433±0.1104  feat=1554  t=98.6s

Processing da